# Chapitre 6 - Exercices 6.1 à 6.6

Régularité de l'estimateur de Nadaraya-Watson, moments du noyau équivalent, variance des
polynômes locaux, choix de la métrique pour des courbes, logistique localement constante
et modèles à coefficients variables.

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from esl.plotting import set_style, COLORS, BLUE, ORANGE, GRAY, SEQ, DIV, TWO_CLASS
from esl import datasets
from esl import ch06_noyaux as nk
set_style()
GREEN, YELLOW, RED = COLORS[2], COLORS[3], COLORS[7]
rng = np.random.RandomState(0)

Données communes : le modèle de la figure 6.1, $Y = \sin(4X) + \varepsilon$,
$X\sim\mathcal U[0,1]$, $\varepsilon\sim\mathcal N(0, 1/3)$, avec ici $N = 40$ points (pour
que les phénomènes locaux soient bien visibles).

In [2]:
N = 40
x = np.sort(rng.uniform(0, 1, N))
y = np.sin(4 * x) + np.sqrt(1 / 3) * rng.normal(size=N)

## Exercice 6.1 : Dérivabilité de l'estimateur de Nadaraya-Watson

**Énoncé.** Montrer que l'estimateur de Nadaraya-Watson à largeur métrique fixe $\lambda$
et noyau gaussien est dérivable. Que peut-on dire avec le noyau d'Epanechnikov ? Et avec le
noyau d'Epanechnikov à largeur adaptative des plus proches voisins $\lambda(x_0)$ ?

**Rappel.** $\hat f(x_0) = \dfrac{N(x_0)}{S(x_0)}$ avec $N(x_0) = \sum_i K_i(x_0)\,y_i$,
$S(x_0) = \sum_i K_i(x_0)$ et $K_i(x_0) = D\big(|x_i - x_0|/h(x_0)\big)$.
Un quotient de fonctions dérivables est dérivable là où le dénominateur ne s'annule pas,
et sa dérivée est $(N'S - NS')/S^2$.

### (a) Noyau gaussien, largeur fixe

Ici $K_i(x_0) = \varphi\big((x_i - x_0)/\lambda\big)$ (comme $\varphi$ est paire, la valeur
absolue ne joue aucun rôle). C'est une composée de fonctions $C^\infty$, donc $K_i$ est
$C^\infty$, ainsi que $N$ et $S$. De plus $S(x_0) > 0$ pour tout $x_0$ car $\varphi > 0$.
Donc $\hat f$ est $C^\infty$ sur $\mathbb R$ entier.

**Calcul de la dérivée.** Comme $\varphi'(t) = -t\varphi(t)$, on obtient par dérivation
composée
$$
K_i'(x_0) = -\frac1\lambda\varphi'\!\left(\frac{x_i - x_0}{\lambda}\right)
= \frac{x_i - x_0}{\lambda^2}\,K_i(x_0).
$$
Notons $w_i = K_i/S$ les poids normalisés. Alors
$$
\hat f'(x_0) = \frac{N'}{S} - \frac{N}{S}\frac{S'}{S}
= \sum_i w_i\frac{x_i - x_0}{\lambda^2}y_i - \hat f(x_0)\sum_i w_i\frac{x_i - x_0}{\lambda^2}
= \frac1{\lambda^2}\sum_i w_i\,(x_i - x_0)\big(y_i - \hat f(x_0)\big) :
$$
c'est une **covariance pondérée** entre $x$ et $y$, divisée par $\lambda^2$.

In [3]:
lam = 0.1
def nw_gauss(t):
    return nk.nadaraya_watson(t, x, y, lam=lam, kernel="gaussian")

t0 = np.linspace(0, 1, 7)
W = nk.kernel_weights(t0, x, lam=lam, kernel="gaussian")
W = W / W.sum(1, keepdims=True)
formula = np.sum(W * (x[None, :] - t0[:, None]) * (y[None, :] - nw_gauss(t0)[:, None]), 1) / lam ** 2
eps = 1e-6
finite = (nw_gauss(t0 + eps) - nw_gauss(t0 - eps)) / (2 * eps)
print("écart max formule / différences finies : %.2e" % np.max(np.abs(formula - finite)))

écart max formule / différences finies : 1.71e-10


### (b) Noyau d'Epanechnikov, largeur fixe

$D(t) = \frac34(1-t^2)$ pour $|t|\le1$, $0$ sinon. On a :
- $D$ est **continue** (elle vaut 0 en $\pm1$ des deux côtés) ;
- $D(|t|) = \frac34(1-t^2)$ près de $t=0$ : la valeur absolue ne crée aucun problème en
  $x_0 = x_i$ ;
- en $t = 1$ : dérivée à gauche $D'(1^-) = -\frac32$, dérivée à droite $D'(1^+) = 0$.
  $D$ **n'est pas dérivable** en $\pm 1$.

Donc $K_i(x_0)$ est continue partout, et dérivable sauf aux deux points
$x_0 = x_i \pm \lambda$, où un point **entre ou sort** de la fenêtre.

**Conséquences pour $\hat f$.**
- $\hat f$ n'est définie que là où $S(x_0) > 0$, c'est-à-dire si au moins un $x_i$ est à
  distance $< \lambda$ de $x_0$ ; elle y est **continue**.
- Elle est dérivable partout sauf, éventuellement, en $x_0 = x_i\pm\lambda$. En un tel point
  $K_i(x_0) = 0$. Calculons le saut de $K_i'$ (dérivée à droite moins dérivée à gauche).
  En $x_0 = x_i + \lambda$ (le point **sort**) : à gauche $K_i = D\big((x_0 - x_i)/\lambda\big)$
  a pour dérivée $D'(1^-)/\lambda = -\frac{3}{2\lambda}$, à droite $K_i \equiv 0$ ; saut
  $+\frac{3}{2\lambda}$. En $x_0 = x_i - \lambda$ (le point **entre**) : à gauche 0, à
  droite $K_i = D\big((x_i - x_0)/\lambda\big)$ de dérivée $-D'(1^-)/\lambda = +\frac{3}{2\lambda}$ ;
  même saut. En reportant dans $\hat f' = (N'S - NS')/S^2 = \sum_j K_j'(y_j - \hat f)/S$
  (les autres $K_j'$ sont continues), le saut de $\hat f'$ vaut
  $$
  \frac{3}{2\lambda}\cdot\frac{y_i - \hat f(x_0)}{S(x_0)},
  $$
  **non nul** sauf si $y_i = \hat f(x_0)$ (cas exceptionnel). Génériquement, $\hat f$ a donc
  un **point anguleux** en chaque $x_i\pm\lambda$.

In [ ]:
lam = 0.15
tt = np.linspace(0.02, 0.98, 4001)
fe = nk.nadaraya_watson(tt, x, y, lam=lam, kernel="epanechnikov")
dfe = np.gradient(fe, tt)
fig, axes = plt.subplots(2, 1, figsize=(9, 6), sharex=True)
axes[0].plot(tt, fe, color=GREEN); axes[0].scatter(x, y, s=10, color=GRAY)
axes[0].set_title("Nadaraya-Watson, Epanechnikov, $\\lambda = 0{,}15$"); axes[0].set_ylabel("$\\hat f$")
axes[1].plot(tt, dfe, color=BLUE, lw=1.2)
for xi in x:
    for s in (-lam, lam):
        if 0.02 < xi + s < 0.98:
            axes[1].axvline(xi + s, color=GRAY, lw=0.4, alpha=0.6)
axes[1].set_title("Dérivée numérique : sauts aux points $x_i \\pm \\lambda$ (traits gris)")
axes[1].set_xlabel("$x_0$"); axes[1].set_ylabel("$\\hat f'(x_0)$")
plt.tight_layout(); plt.show()

Vérifions la formule du saut en un point $x_0 = x_i + \lambda$ :

In [ ]:
i = 15
xc = x[i] + lam
h = 1e-7
left = (nk.nadaraya_watson([xc - h], x, y, lam=lam)[0] - nk.nadaraya_watson([xc - 2 * h], x, y, lam=lam)[0]) / h
right = (nk.nadaraya_watson([xc + 2 * h], x, y, lam=lam)[0] - nk.nadaraya_watson([xc + h], x, y, lam=lam)[0]) / h
S = nk.kernel_weights([xc], x, lam=lam)[0].sum()
fx = nk.nadaraya_watson([xc], x, y, lam=lam)[0]
print("saut observé  : %.4f" % (right - left))
print("saut théorique : %.4f" % (3 / (2 * lam) * (y[i] - fx) / S))

### (c) Epanechnikov à largeur adaptative (k plus proches voisins)

Maintenant $h(x_0) = |x_0 - x_{[k]}|$, distance au $k$-ième voisin.

- **Continuité de $h$** : $h(x_0)$ est la $k$-ième plus petite des fonctions continues
  $|x_0 - x_i|$, donc continue ; elle est affine par morceaux, de pente $\pm1$.
- **Le $k$-ième voisin a un poids nul** : $|x_{[k]} - x_0|/h(x_0) = 1$ et $D(1) = 0$. En
  pratique la moyenne porte donc sur $k - 1$ points.
- Quand le $k$-ième voisin **change** (un point $a$ à gauche et un point $b$ à droite sont
  à égale distance), les deux points concernés ont un poids nul de part et d'autre : pas de
  saut, $\hat f$ reste **continue** (et elle est toujours définie : $S > 0$ dès que $k\ge 2$
  sans ex æquo).
- Mais en ce point la pente de $h$ passe de $+1$ à $-1$ (ou l'inverse) : $h$ a un **point
  anguleux**. Tous les poids $K_j = D(|x_j - x_0|/h(x_0))$ des points intérieurs ont alors
  une dérivée qui saute, et $\hat f'$ saute génériquement.

Conclusion : avec la largeur adaptative, $\hat f$ est continue mais présente des points
anguleux aux points de changement du $k$-ième voisin (milieux $(a + b)/2$ de paires de
points d'apprentissage). Même un noyau régulier (tri-cube, gaussien tronqué) ne suffirait
pas à rendre $\hat f$ dérivable, puisque c'est $h(x_0)$ lui-même qui ne l'est pas.